In [2]:
import numpy as np

## Overview

This notebook includes:

- Implementation of the Categorical Cross-Entropy Loss
- Implementation of Accuracy calculation
- Full neural network implementation developed so far

Case1: When class Targets are different number such as 0,1,2, Here:

        0-Red, 1-Green, 2-Blue

In [10]:
softmax_outputs=np.array([[0.7,0.1,0.2],
                          [0.1,0.5,0.4],
                          [0.02,0.9,0.08]])
class_targets=[0,1,1]
#Indexing
print(softmax_outputs[[0,1,2], class_targets])

[0.7 0.5 0.9]


In [4]:
print(-np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
    ]))
neg_log=-np.log(softmax_outputs[range(len(softmax_outputs)),class_targets])
average_loss=np.mean(neg_log)
print(average_loss)

[0.35667494 0.69314718 0.10536052]
0.38506088005216804


In [5]:
# Trying Indexing through loop
# a=[]
# for i in range(len(softmax_outputs)):
#     a.append(softmax_outputs[[i],class_targets[i]])
# a=np.array(a)
# print(a)



Case2: Targets are ONE-HOT Encoded, Here:

        Red-[1 0 0],Green-[0 1 0], Blue-[ 0 0 1]

if Data is one hot encoded, how to extract the relevant predictions

In [ ]:
y_true_check=np.array([
    [0,1,0],
    [1,0,0],
    [0,0,1]
])

y_pred_clipped_check=np.array([
    [0.2,0.7,0.1],
    [0.8,0.1,0.1],
    [0.1,0.2,0.7]
])

A=y_true_check*y_pred_clipped_check
B=np.sum(A, axis=1)
print(B)

[0.7 0.8 0.7]


(3, 3)

In [7]:
C= -np.log(B)
print(C)

[0.35667494 0.22314355 0.35667494]


Note: We need to prevent very large or very small values when we take log. Hence, we need to clip our predcitions

In [ ]:
# Implementing the loss class

class Loss:
    # Calculates the data and regularization losses
    def calculate(self, output,y):
        #Calculate sample losses
        sample_losses=self.forward(output,y)
        #Calculate mean loss
        data_loss=np.mean(sample_losses)
        #return Loss
        return data_loss

#we implement the class loss just for the sake of simplicity
#So we can also see weight values

In [14]:
# Implemnting the categorical cross entropy class

class Loss_CategoricalCrossentropy(Loss):
    #forward pass
    def forward(self,y_pred,y_true):
        samples=len(y_pred)
        #Clip data to prevent divisions by 0
        #Clip both sides to not drag mean towards any value
        y_pred_clipped=np.clip(y_pred, 1e-7,1-1e-7)
        if len(y_true.shape)==1:
            correct_confidences=y_pred_clipped[
                range(samples),
                y_true
            ]
            #Mask values -only for one-hot encoded labels
        elif len(y_true.shape)==2:
            correct_confidences = np.sum(
                y_pred_clipped*y_true,
                axis=1
            )
        #losses
        negative_log_likelihoods=-np.log(correct_confidences)
        return negative_log_likelihoods



In [15]:
softmax_outputs=np.array([[0.7,0.1,0.2],
                          [0.1,0.5,0.4],
                          [0.02,0.9, 0.8]])
class_targets=np.array([[1,0,0],
                        [0,1,0],
                        [0,1,0]])
loss_function=Loss_CategoricalCrossentropy()
loss=loss_function.calculate(softmax_outputs, class_targets)
print(loss)

0.38506088005216804


In [ ]:
#Introducing accuracy

softmax_outputs=np.array([[0.7, 0.2,0.1],
                          [0.1,0.5,0.4],
                          [0.02, 0.9, 0.08]])
class_targets=np.array([0,1,1])

predictions=np.argmax(softmax_outputs, axis=1)

#if targets are one hot encoded -convert them
if len(class_targets.shape)==2:
    class_targets=np.argmax(class_targets, axis=1)
    
# true evaluates to 1; False to 0
accuracy=np.mean(predictions==class_targets)
print('acc:',accuracy)

[0 1 1]
acc: 1.0


Full code upto this point

In [ ]:
from nnfs.datasets import spiral_data
import numpy as np
import nnfs
import matplotlib.pyplot as plt

In [27]:
# Dense Layer class
import numpy as np
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()
#Dense layer

class Layer_dense:
    #Layer intialization
    def __init__(self, n_inputs, n_neurons):
        self.weights=0.01*np.random.randn(n_inputs, n_neurons)
        self.biases=np.zeros((1,n_neurons))

    #Forward pass
    def forward(self, inputs):
        #Calculate output values from inputs, weights, biases
        self.output=np.dot(inputs, self.weights)+self.biases

In [28]:
# Relu activation 
class Activation_Relu:
    #forward pass
    def forward(self,input):
        #Calculate output values from input
        self.output=np.maximum(0,input)

In [29]:
#Softmax activation

class Activation_Softmax:
    # Forward Pass
    def forward(self, inputs):
        #Get unnormalized probabilities
        exp_values=np.exp(inputs-np.max(inputs, axis=1, keepdims=True))
        #Normalize them for each sample
        probabilities=exp_values/np.sum(exp_values, axis=1, keepdims=True)
        self.output=probabilities

In [30]:
# Implementing the loss class

class Loss:
    # Calculates the data and regularization losses
    def calculate(self, output,y):
        #Calculate sample losses
        sample_losses=self.forward(output,y)
        #Calculate mean loss
        data_loss=np.mean(sample_losses)
        #return Loss
        return data_loss

#we implement the class loss just for the sake of simplicity
#So we can also see weight values

In [23]:
# Implemnting the categorical cross entropy class

class Loss_CategoricalCrossentropy(Loss):
    #forward pass
    def forward(self,y_pred,y_true):
        samples=len(y_pred)
        #Clip data to prevent divisions by 0
        #Clip both sides to not drag mean towards any value
        y_pred_clipped=np.clip(y_pred, 1e-7,1-1e-7)
        if len(y_true.shape)==1:
            correct_confidences=y_pred_clipped[
                range(samples),
                y_true
            ]
            #Mask values -only for one-hot encoded labels
        elif len(y_true.shape)==2:
            correct_confidences = np.sum(
                y_pred_clipped*y_true,
                axis=1
            )
        #losses
        negative_log_likelihoods=-np.log(correct_confidences)
        return negative_log_likelihoods


In [41]:
X,y=spiral_data(samples=100,classes=3)
dense1=Layer_dense(2,3)
activation1=Activation_Relu()
dense2=Layer_dense(3,3)
activation2=Activation_Softmax()
#Create loss function
loss_function=Loss_CategoricalCrossentropy()

#performing forward pass
dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
activation2.forward(dense2.output)
print(activation2.output[:5])
loss=loss_function.calculate(activation2.output,y)
print('loss:',loss)

#Calculate accuracy from output of acitvation2 and targets

predictions=np.argmax(activation2.output, axis=1)
if len(y.shape)==2:
    y=np.argmax(y, axis=1)
accuracy=np.mean(predictions==y)

print('acc:', accuracy)


[[0.33333334 0.33333334 0.33333334]
 [0.33333337 0.33333337 0.3333332 ]
 [0.33333406 0.33333424 0.33333164]
 [0.33333427 0.33333445 0.33333123]
 [0.33333403 0.33333415 0.33333182]]
loss: 1.0986098
acc: 0.4
